In [1]:
with open("../data/clean_logs_linux.txt", encoding="latin-1") as f:
    logs = [line.strip() for line in f]



clean_logs = list(set(logs))
print(clean_logs)
print(len(clean_logs))

['kernel process syslogd is using obsolete setsockopt so_bsdcompat', 'kernel audit avc denied name_bind for pid exe usr sbin xinetd scontext system_u system_r kernel_t tcontext system_u object_r biff_port_t tclass udp_socket', 'gdm pam_unix bad username', 'ftpd connection from www zajelgate net at sun jan', 'sshd pam_unix authentication failure logname uid euid tty nodevssh ruser rhost dimenoc com user root', 'ftpd connection from adsl dsl pacbell net at mon jul', 'smartd smartd version copyright c bruce allen', 'kernel audit avc denied name_bind for pid exe usr sbin ntpd scontext system_u system_r kernel_t tcontext system_u object_r ntp_port_t tclass udp_socket', 'kernel d_instantiate', 'kernel agpgart agp aperture is', 'sshd pam_unix session closed for user test', 'sshd pam_unix check pass user unknown', 'kernel mapped trampoline to', 'login pam_unix authentication failure logname login uid euid tty ruser rhost', 'ftpd connection from dsl chn static touchtelindia net at sun jul', 'ke

In [2]:
with open("../data/clean_logs_hdfs.txt", encoding="latin-1") as f:
    logs = [line.strip() for line in f]

print(logs[:10])
print(len(set(logs)))

['Deleting block file path', 'PacketResponder block Exception millis timeout while waiting for channel to be ready read. ch [connected local=path remote=path]', 'BLOCK Removing block from neededReplications as it does not belong to any file.', 'Receiving empty packet for block', 'BLOCK path block', 'PacketResponder for block terminating', 'Adding an already existing block', 'ip Exception writing block to mirror', 'writeBlock block received exception Interruped while waiting for IO on channel [connected local=path remote=path]. millis timeout left.', 'PacketResponder block Exception Connection reset by peer']
49


In [3]:
with open("../data/clean_logs_apache.txt") as f:
    logs = [line.strip() for line in f]

# print(set(logs))
print(len(list(set(logs))))

36


In [4]:
import os

# Change the working directory to the parent (root) folder
if os.path.basename(os.getcwd()) == 'training':
    os.chdir('..')

print(f"Current Working Directory: {os.getcwd()}")

# Now imports and data paths will work as if you were in the root
from read_all_logs import read_Logs, deduplicate_logs

raw_logs = read_Logs()
unique_logs = deduplicate_logs(raw_logs)
print(unique_logs)

Current Working Directory: D:\Code\nlp-tutorials\semantic-search-logs
Total logs before deduplication: 1069
Total logs after deduplication: 1069
[{'text': 'error env createbean2 factory error creating worker jni onstartup worker jni onstartup', 'source': 'Apache'}, {'text': 'notice jk2_init found child <NUM> in scoreboard slot <NUM>', 'source': 'Apache'}, {'text': 'notice ldap built with openldap ldap sdk', 'source': 'Apache'}, {'text': 'error config update can t create worker jni onstartup', 'source': 'Apache'}, {'text': 'error env createbean2 factory error creating worker jni onshutdown worker jni onshutdown', 'source': 'Apache'}, {'text': 'error config update can t create vm', 'source': 'Apache'}, {'text': 'error request failed error reading the headers', 'source': 'Apache'}, {'text': 'error file does not exist <PATH>', 'source': 'Apache'}, {'text': 'error attempt to serve directory <PATH>', 'source': 'Apache'}, {'text': 'error urimap mapuri uri must start with', 'source': 'Apache'}

In [6]:
import pandas as pd

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(unique_logs)

# Display the first few rows
print(df.head())


                                                text  source
0  error env createbean2 factory error creating w...  Apache
1  notice jk2_init found child <NUM> in scoreboar...  Apache
2           notice ldap built with openldap ldap sdk  Apache
3  error config update can t create worker jni on...  Apache
4  error env createbean2 factory error creating w...  Apache


In [7]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

C:\Users\Ameer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [log["text"] for log in unique_logs]

embeddings = model.encode(texts, show_progress_bar=True)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3437.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|█████████████████████████████████████████████████████████████████████████| 34/34 [00:07<00:00,  4.40it/s]


In [9]:
df['embeddings'] = list(embeddings)

# Check the result
print(df.head())

                                                text  source  \
0  error env createbean2 factory error creating w...  Apache   
1  notice jk2_init found child <NUM> in scoreboar...  Apache   
2           notice ldap built with openldap ldap sdk  Apache   
3  error config update can t create worker jni on...  Apache   
4  error env createbean2 factory error creating w...  Apache   

                                          embeddings  
0  [-0.110188186, -0.012358566, -0.009629513, -0....  
1  [-0.00463686, 0.00038357842, -0.07665329, 0.00...  
2  [-0.022290597, 0.037510343, -0.03316659, 0.004...  
3  [-0.10561246, 0.014960397, -0.0055450043, -0.0...  
4  [-0.113687284, -0.007040228, -0.011687007, -0....  


In [10]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
df['cluster'] = clustering.labels_

In [11]:
df.head

<bound method NDFrame.head of                                                    text  source  \
0     error env createbean2 factory error creating w...  Apache   
1     notice jk2_init found child <NUM> in scoreboar...  Apache   
2              notice ldap built with openldap ldap sdk  Apache   
3     error config update can t create worker jni on...  Apache   
4     error env createbean2 factory error creating w...  Apache   
...                                                 ...     ...   
1064  sshd pam_unix authentication failure logname u...   Linux   
1065                        rc stopping squid succeeded   Linux   
1066           squid squid parent child process started   Linux   
1067                                  kernel dump_stack   Linux   
1068  sshd pam_unix authentication failure logname u...   Linux   

                                             embeddings  cluster  
0     [-0.110188186, -0.012358566, -0.009629513, -0....        0  
1     [-0.00463686, 0.000383578

In [12]:
df['cluster'].value_counts()

cluster
59     171
61     134
62      97
64      30
56      14
      ... 
439      1
440      1
441      1
442      1
443      1
Name: count, Length: 444, dtype: int64

In [13]:
df.to_csv('data/clustered_logs.csv', index=False)

print("File saved! You can now download it from the VS Code file explorer and upload it here.")

File saved! You can now download it from the VS Code file explorer and upload it here.


In [11]:
import sys
!{sys.executable} -m pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 3.4 MB/s eta 0:00:06
   -- ------------------------------------- 1.3/18.9 MB 2.8 MB/s eta 0:00:07
   --- ------------------------------------ 1.8/18.9 MB 2.8 MB/s eta 0:00:07
   ----- ---------------------------------- 2.6/18.9 MB 3.0 MB/s eta 0:00:06
   ------- -------------------------------- 3.4/18.9 MB 3.1 MB/s eta 0:00:06
   -------- ------------------------------- 4.2/18.9 MB 3.2 MB/s eta 0:00:05
   ---------- ----------------------------- 5.0/18.9 MB 3.3 MB/s eta 0:00:05
   ------------ --------------------------- 5.8/18.9 MB 3.4 MB/s eta 0:00:04
   ------------- -------------------------- 6.6/18.9 MB 3.4 MB/s eta 0:00:04
   --------------- ------------------------ 7.3/18.9 MB 3.5 MB/s eta 0:00:04
   ----------------- ---------------------- 8.1/18.9 MB 3.5 MB/s eta 0:00:04
   ----------

In [14]:
import faiss
index = faiss.read_index("models/log_index.faiss")

In [16]:
import os
print(f"Current Directory: {os.getcwd()}")
print(f"Files here: {os.listdir('.')}")

Current Directory: D:\Code\nlp-tutorials\semantic-search-logs\training
Files here: ['.ipynb_checkpoints', 'log_identity.ipynb']


In [17]:
index = faiss.read_index("../models/log_index.faiss")

In [20]:
import numpy as np

# Reconstruct the vector at index 0
first_row = index.reconstruct(10)

print("First row vector:")
print(first_row)

First row vector:
[ 6.44080713e-02  2.73998007e-02  2.00820602e-02 -1.03080750e-01
 -4.60359035e-03 -2.13050768e-02 -6.48978502e-02  4.61940616e-02
 -6.85120896e-02  4.21216860e-02  6.41347021e-02 -7.27981552e-02
  1.53376907e-03  5.49378917e-02 -2.50785798e-02  1.78912394e-02
 -6.23187236e-02  1.93251278e-02  2.93801352e-02  2.95278095e-02
  3.03999241e-02  1.50361797e-02  1.29680540e-02 -7.58361369e-02
  1.33807538e-02  5.40192844e-03 -8.61763675e-03 -2.28929259e-02
 -1.19802905e-02 -1.21594327e-04  4.67583425e-02 -7.03266710e-02
  1.34146186e-02 -2.87568308e-02  1.03738151e-01  1.71067014e-01
 -1.01015139e-02 -7.93294832e-02 -4.45177080e-03  1.63526386e-02
  5.95569536e-02 -4.62839380e-02  4.97065205e-03 -3.83835696e-02
  2.73944233e-02  6.19055741e-02  5.96245639e-02  3.03898342e-02
  6.40192116e-03  5.14253713e-02 -1.04765303e-01  3.06280423e-02
  6.70383824e-03 -3.73612195e-02  1.03128627e-02 -3.90178598e-02
 -4.66129072e-02  4.59354110e-02  1.23589393e-02 -1.42180093e-03
 -1.442

In [19]:
print(f"Total rows in index: {index.ntotal}")
print(f"Vector dimensions: {index.d}")

Total rows in index: 1065
Vector dimensions: 384
